In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_building_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateBuildingId String, SourceSystem String, IngestionTimestamp timestamp, BatchId String, RealEstateBuildingKey string, RealEstateBuildingType string,"
        #     "RealEstateKey string, RealEstateCepalKey String, RealEstateStatusCode String, RealEstateStatus String, SourceTableName String, RealEstateComments String, RealEstateActive String,"
        #     "RealEstateDescription String, RealEstateSubCategoryCode String, RealEstateSubCategory String, RealEstateCategoryCode String, RealEstateCategory String,"
        #     "RealEstateAddress1 String, RealEstateAddress2 String, RealEstateBuildingNumber String, RealEstateEntranceAddress String, RealEstatePostalCode String,"
        #     "RealEstateGeolocation String, RealEstateLocationLambda double, RealEstateLocationPhi double, RealEstateEncumbrances String, RealEstateEncumbrancesStatus String, RealEstateEncumbrancesDetails String, "
        #     "RealEstateLandArea double, RealEstateContractArea double, RealEstateNetArea double, RealEstateGlaArea double,  RealEstateOccupancyTypeCode string, RealEstateOccupancyType string, "
        #     "RealEstateOccupancyRate double, RealEstateSurfaceArea double, RealEstatePlottedArea double,"
        #     "RealEstatePlottedDate date, RealEstateAcquisitionValue decimal(19,4), RealEstateBookValue decimal(19,4), RealEstateApprovedIASValue decimal(19,4), RealEstateFiscalValue decimal(19,4),"
        #     "RealEstateMarketValue decimal(19,4), RealEstateEvaluationDate date, RealEstateEvaluationTypeCode String, RealEstateEvaluationType String, RealEstateForcedSaleValue decimal(19,4),"
        #     "RealEstateSaleAmount decimal(19,4), RealEstateSaleVATAmount decimal(19,4), RealEstateSaleTotalAmount decimal(19,4), RealEstateSaleDate date, RealEstateDateEntered date,"
        #     "RealEstateFirstOfferPrice decimal(19,4), RealEstateMinimumPrice decimal(19,4), RealEstateMaximumPrice decimal(19,4), RealEstateAuctionAmount decimal(19,4), RealEstateConstructionPercentage String,"
        #     "RealEstateLegalIssuesFlag boolean, RealEstateValueDate date, RealEstateGroupMarketValueDate date, RealEstateValueTypeCode String, RealEstateValueType String, RealEstateGroupMarketValue decimal(19,2),"
        #     "RealEstateGroupMarketValueTypeCode String, RealEstateGroupMarketValueType String, RealEstateConstructionYear String, RealEstateLandRegistryCode String, RealEstateLandRegistryOffice String,"
        #     "RealEstateLandRegistry String, RealEstateUsageCodeTypeCode String, RealEstateUsageCodeType String, RealEstateSiteCode String, RealEstatePrimaryLandLocation String, RealEstateTotalTax decimal(19,2),"
        #     "RealEstateOperatingCost decimal(19,2), RealEstateArea String, RealEstateCity String, RealEstateIsEAPFlag boolean, RealEstateDistrict String, RealEstateDivision String,"
        #     "RealEstateMunicipality String, RealEstateCounty String, RealEstateMarketabilityCode String, RealEstateKaekCode String, RealEstateKaekOffice String, RealEstateHotelFlag boolean,"
        #     "RealEstateHotelStars String, RealEstateHotelRooms int, RealEstateSpecialConditions String, RealEstateAuctionDate timestamp, RealEstateAuctionCode String, RealEstatePlaceCode String,"
        #     "RealEstateConstructionStatusCode String, RealEstateConstructionStatus String, RealEstateBuildingName String, RealEstateBuildingActive String, RealEstateBuildingLandRegistryUnit String,"
        #     "RealEstateBuildingLandBookNumber String, RealEstateBuildingOldLandRegistryUnit String, RealEstatePropertyBuildingNumber String, RealEstateBuildingOldLandBookNumber String,"
        #     "RealEstateBuildingPreviousRegistryUnit String, RealEstateBuildingTopographicNumber String, RealEstateBuildingBuildingTypeCode String, RealEstateBuildingBuildingType String,"
        #     "RealEstateBuildingDescription String, RealEstateBuildingFloorRawValue double, RealEstateBuildingContractArea double, RealEstateBuildingNetArea double, RealEstateBuildingSurfaceArea2 double,"
        #     "RealEstateBuildingGrossRentableArea double, RealEstateBuildingOccupancyRate double, RealEstateBuildingRenovationYear int, RealEstateBuildingPermitNumber String, RealEstateLandOwnership string,"
        #     "RealEstateBuildingPermitDate timestamp, RealEstateBuildingFacadeArea double, RealEstateBuildingFreeHeight double, RealEstateBuildingTechnicalNotes String, RealEstateBuildingOtherAuxiliaryArea double,"
        #     "RealEstateBuildingParkingSpace int, RealEstateBuildingStorageArea double, RealEstateBuildingLoftArea double, RealEstateBuildingDetachmentArea String, RealEstateBuildingPowerNumber String,"
        #     "RealEstateBuildingDateEntered timestamp, RealEstateBuildingEnteredBy int, RealEstateBuildingAreaTitle double, RealEstateBuildingAreaLicense double, RealEstateBuildingAreaSettled double,"
        #     "RealEstateBuildingUsageCode String, RealEstateBuildingParkingArea double, RealEstateBuildingWarehouseNumber int, RealEstateIsEligibleFlag boolean, RealEstateCountry string,"
        #     "CreatedDate string, CreatedBy string, LastUpdatedDate string, LastUpdatedBy string,  LocationTrackingCode string, LocationTracking string, BusinessDate date, RealEstateBuildingFloorRawValueDescription string"

        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateBuilding"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== prproperty ==================

        prproperty_extra_filters = None

        if prproperty_extra_filters is not None:
            prproperty_extra_filters.append(date_filter)
            prproperty_final_filters = " AND ".join(prproperty_extra_filters)
        else:
            prproperty_final_filters = date_filter

        prproperty_selected_fields = [
            "id",
            "land_ownership",
            "active",
            "status_code",
            "comments",
            "descr",
            "cat_descr",
            "category",
            "address",
            "address_2",
            "building_no",
            "entrance",
            "zip",
            "geolocation",
            "encumbrances",
            "encumbrances_status",
            "encumbrances_details",
            "land_area",
            "contract_area",
            "net_area",
            "gla_area",
            "occupancy_rate",
            "fixed_area",
            "plotted_area",
            "plotted_date",
            "acquisition_value",
            "eligible_property",
            "place_code",
            "book_value",
            "approved_ias_value",
            "fiscal_value",
            "market_value",
            "eval_date",
            "eval_type",
            "forced_sale_value",
            "auction_date",
            "sale_amount",
            "sale_vat_amount",
            "sale_total_amount",
            "sold_date",
            "date_entered",
            "first_offer",
            "minimum_price",
            "maximum_price",
            "auction_amount",
            "construction_perc",
            "has_legal",
            "ownership_descr",
            "value_date",
            "group_market_value_date",
            "auction_code",
            "value_type",
            "group_market_value",
            "group_value_type",
            "year_made",
            "kaek",
            "kaek_office",
            "ktimatologio_id",
            "usage_code",
            "usage_code_descr",
            "site_code",
            "prland",
            "total_tax",
            "operating_cost",
            "area",
            "city",
            "cepal_key",
            "place_perif_enothta",
            "place_perifereia",
            "place_dhmos",
            "place_dhm_enothta",
            "marketability_code",
            "hotel_flag",
            "hotel_stars",
            "hotel_rooms",
            "special_conditions",
            "location_lambda",
            "location_phi",
            "country_code",
            "businessdate",
            "location_tracking",
            "landuse_id",
            "insurance_cat",

        ]

        prproperty = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty",
            prproperty_final_filters,
            prproperty_selected_fields,
        )

        # ================== prproperty_building ==================

        prproperty_building_extra_filters = None

        if prproperty_building_extra_filters is not None:
            prproperty_building_extra_filters.append(date_filter)
            prproperty_building_final_filters = " AND ".join(
                prproperty_building_extra_filters
            )
        else:
            prproperty_building_final_filters = date_filter

        prproperty_building_selected_fields = [
            "id",
            "property_id",
            "name",
            "active",
            "cad_no",
            "cf_no",
            "old_cad_no",
            "number",
            "old_cf_no",
            "previous_cad",
            "topographic_no",
            "asset_type",
            "descr",
            "floor_id",
            "contract_area",
            "net_area",
            "fixed_area_2",
            "gla_area",
            "occupancy_rate",
            "renovation_year",
            "permit_no",
            "permit_date",
            "tte_facade",
            "tte_freeheight",
            "tte_technicalnotes",
            "tte_otherauxiliaryarea",
            "tte_parkingspaces",
            "tte_storagearea",
            "tte_loftarea",
            "tte_PropertyDetachment",
            "power_no",
            "date_entered",
            "entered_by",
            "area_title",
            "area_license",
            "area_settled",
            "usage",
            "parking_area",
            "warehouse_no",
            "property_id",
            "businessdate",
        ]

        prproperty_building = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prproperty_building",
            prproperty_building_final_filters,
            prproperty_building_selected_fields,
        )

        # ================== prlanduse===============

        prlanduse_extra_filters = None

        if prlanduse_extra_filters is not None:
            prlanduse_extra_filters.append(date_filter)
            prlanduse_final_filters = " AND ".join(prlanduse_extra_filters)
        else:
            prlanduse_final_filters = date_filter

        prlanduse_selected_fields = [
            "id",
            "descr",
        ]

        prlanduse = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prlanduse",
            prlanduse_final_filters,
            prlanduse_selected_fields,
        )
        # ==============================thesis_prpropertytemp====================
        thesis_prpropertytemp_extra_filters = None

        if thesis_prpropertytemp_extra_filters is not None:
            thesis_prpropertytemp_extra_filters.append(date_filter)
            thesis_prpropertytemp_final_filters = " AND ".join(
                thesis_prpropertytemp_extra_filters
            )
        else:
            thesis_prpropertytemp_final_filters = date_filter

        thesis_prpropertytemp_selected_fields = [
            "ID",
            "status_code",
            "comments",
            "descr",
            "cat_code",
            "table_name",
            "address",
            "address_2",
            "building_no",
            "place_code",
            "zip",
            "location_lambda",
            "location_phi",
            "land_area",
            "net_area",
            "date_entered",
            "kaek",
            "usage_code",
            "area",
            "city",
            "floor_id",
            "BusinessDate",
        ]

        thesis_prpropertytemp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertytemp",
            thesis_prpropertytemp_final_filters,
            thesis_prpropertytemp_selected_fields,
        )

        # ==============================thesis_prpropertybuildingtemp====================

        thesis_prpropertybuildingtemp_extra_filters = None

        if thesis_prpropertybuildingtemp_extra_filters is not None:
            thesis_prpropertybuildingtemp_extra_filters.append(date_filter)
            thesis_prpropertybuildingtemp_final_filters = " AND ".join(
                thesis_prpropertybuildingtemp_extra_filters
            )
        else:
            thesis_prpropertybuildingtemp_final_filters = date_filter

        thesis_prpropertybuildingtemp_selected_fields = [
            "ID",
            "property_temp_id",
            "date_entered",
            "entered_by",
            "tte_otherauxiliaryarea",
            "tte_storagearea",
            "parking_area",
            "floor_id",
            "net_area",
            "BusinessDate",
        ]

        thesis_prpoperty_building_temp = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prpropertybuildingtemp",
            thesis_prpropertybuildingtemp_final_filters,
            thesis_prpropertybuildingtemp_selected_fields,
        )

        # ================== Joins ==================
        prproperty_prproperty_building_join_condition = (
            prproperty["id"] == prproperty_building["property_id"]
        )

        prproperty_prlanduse_join_condition = (
            prproperty["landuse_id"] == prlanduse["id"]
        )

        target_table_thesis = prproperty.join(
            prproperty_building, prproperty_prproperty_building_join_condition, "left"
        ).join(prlanduse, prproperty_prlanduse_join_condition, "left")

        prproperty_building_primary_key_fields = [
            prproperty_building.id,
            prproperty.id,
            lit("thesis"),
        ]

        RealEstateBuildingKey_primary_key_fields = [
            prproperty_building.id,
            prproperty.id,
        ]

        target_table_thesis = (
            target_table_thesis.withColumn(
                "RealEstateBuildingId",
                sha2(concat_ws(";", *prproperty_building_primary_key_fields), 256),
            )
            .withColumn(
                "RealEstateBuildingKey",
                concat_ws(";", *RealEstateBuildingKey_primary_key_fields),
            )
            .withColumn(
                "LocationTracking",
                when(prproperty.location_tracking == "01", "Εντοπίσιμο")
                .when(prproperty.location_tracking == "02", "Μη εντοπίσιμο")
                .when(prproperty.location_tracking == "03", "Μη επεξεργασμένο"),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_thesis = target_table_thesis.select(
            target_table_thesis.RealEstateBuildingId,
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table_thesis.RealEstateBuildingKey,
            lit("Real Estate").alias("RealEstateBuildingType"),
            prproperty.id.cast("string").alias("RealEstateKey"),
            prproperty.cepal_key.alias("RealEstateCepalKey"),
            prproperty.status_code.alias("RealEstateStatusCode"),
            prproperty.status_code.alias("RealEstateStatus"),
            lit(None).cast("String").alias("SourceTableName"),
            prproperty.comments.alias("RealEstateComments"),
            when(prproperty.active == 1, "Active")
            .when(prproperty.active == 0, "Inactive")
            .otherwise("NULL")
            .alias("RealEstateActive"),
            prproperty.descr.alias("RealEstateDescription"),
            prproperty.cat_descr.alias("RealEstateSubCategoryCode"),
            prproperty.cat_descr.alias("RealEstateSubCategory"),
            prproperty.category.alias("RealEstateCategoryCode"),
            prproperty.category.alias("RealEstateCategory"),
            prproperty.insurance_cat.cast("string").alias("RealEstateInsuranceCategory"),
            prproperty.address.alias("RealEstateAddress1"),
            prproperty.address_2.alias("RealEstateAddress2"),
            prproperty.building_no.alias("RealEstateBuildingNumber"),
            prproperty.entrance.alias("RealEstateEntranceAddress"),
            prproperty.zip.alias("RealEstatePostalCode"),
            prproperty.geolocation.alias("RealEstateGeolocation"),
            prproperty.location_lambda.alias("RealEstateLocationLambda"),
            prproperty.location_phi.alias("RealEstateLocationPhi"),
            prproperty.encumbrances.alias("RealEstateEncumbrances"),
            prproperty.encumbrances_status.alias("RealEstateEncumbrancesStatus"),
            prproperty.encumbrances_details.alias("RealEstateEncumbrancesDetails"),
            prproperty.land_area.alias("RealEstateLandArea"),
            prproperty.contract_area.alias("RealEstateContractArea"),
            prproperty.net_area.alias("RealEstateNetArea"),
            prproperty.gla_area.alias("RealEstateGlaArea"),
            prproperty.landuse_id.cast("string").alias("RealEstateOccupancyTypeCode"),
            prlanduse.descr.alias("RealEstateOccupancyType"),
            prproperty.occupancy_rate.alias("RealEstateOccupancyRate"),
            prproperty.fixed_area.alias("RealEstateSurfaceArea"),
            prproperty.plotted_area.alias("RealEstatePlottedArea"),
            prproperty.plotted_date.alias("RealEstatePlottedDate"),
            prproperty.acquisition_value.alias("RealEstateAcquisitionValue"),
            prproperty.book_value.alias("RealEstateBookValue"),
            prproperty.approved_ias_value.alias("RealEstateApprovedIASValue"),
            prproperty.fiscal_value.alias("RealEstateFiscalValue"),
            prproperty.market_value.alias("RealEstateMarketValue"),
            prproperty.eval_date.alias("RealEstateEvaluationDate"),
            prproperty.eval_type.alias("RealEstateEvaluationTypeCode"),
            prproperty.eval_type.alias("RealEstateEvaluationType"),
            prproperty.forced_sale_value.alias("RealEstateForcedSaleValue"),
            prproperty.sale_amount.alias("RealEstateSaleAmount"),
            prproperty.sale_vat_amount.alias("RealEstateSaleVATAmount"),
            prproperty.sale_total_amount.alias("RealEstateSaleTotalAmount"),
            prproperty.sold_date.alias("RealEstateSaleDate"),
            prproperty.date_entered.alias("RealEstateDateEntered"),
            prproperty.first_offer.alias("RealEstateFirstOfferPrice"),
            prproperty.minimum_price.alias("RealEstateMinimumPrice"),
            prproperty.maximum_price.alias("RealEstateMaximumPrice"),
            prproperty.auction_amount.alias("RealEstateAuctionAmount"),
            prproperty.construction_perc.alias("RealEstateConstructionPercentage"),
            prproperty.has_legal.alias("RealEstateLegalIssuesFlag"),
            prproperty.value_date.alias("RealEstateValueDate"),
            prproperty.group_market_value_date.alias("RealEstateGroupMarketValueDate"),
            prproperty.value_type.alias("RealEstateValueTypeCode"),
            prproperty.value_type.alias("RealEstateValueType"),
            prproperty.group_market_value.alias("RealEstateGroupMarketValue"),
            prproperty.group_value_type.alias("RealEstateGroupMarketValueTypeCode"),
            prproperty.group_value_type.alias("RealEstateGroupMarketValueType"),
            prproperty.year_made.alias("RealEstateConstructionYear"),
            prproperty.kaek.alias("RealEstateLandRegistryCode"),
            prproperty.kaek_office.alias("RealEstateLandRegistryOffice"),
            prproperty.ktimatologio_id.alias("RealEstateLandRegistry"),
            prproperty.usage_code.alias("RealEstateUsageCodeTypeCode"),
            prproperty.usage_code_descr.alias("RealEstateUsageCodeType"),
            prproperty.site_code.alias("RealEstateSiteCode"),
            prproperty.prland.alias("RealEstatePrimaryLandLocation"),
            prproperty.total_tax.alias("RealEstateTotalTax"),
            prproperty.operating_cost.alias("RealEstateOperatingCost"),
            prproperty.area.alias("RealEstateArea"),
            prproperty.city.alias("RealEstateCity"),
            when(prproperty.cepal_key.like("RES%"), True)
            .otherwise(False)
            .alias("RealEstateIsEAPFlag"),
            prproperty.place_perif_enothta.alias("RealEstateDistrict"),
            prproperty.place_perifereia.alias("RealEstateDivision"),
            prproperty.place_dhmos.alias("RealEstateMunicipality"),
            prproperty.place_dhm_enothta.alias("RealEstateCounty"),
            prproperty.marketability_code.alias("RealEstateMarketabilityCode"),
            prproperty.kaek.alias("RealEstateKaekCode"),
            prproperty.kaek_office.alias("RealEstateKaekOffice"),
            prproperty.hotel_flag.alias("RealEstateHotelFlag"),
            prproperty.hotel_stars.alias("RealEstateHotelStars"),
            prproperty.hotel_rooms.alias("RealEstateHotelRooms"),
            prproperty.special_conditions.alias("RealEstateSpecialConditions"),
            prproperty.auction_date.alias("RealEstateAuctionDate"),
            prproperty.auction_code.alias("RealEstateAuctionCode"),
            prproperty.place_code.alias("RealEstatePlaceCode"),
            prproperty.construction_perc.alias("RealEstateConstructionStatusCode"),
            prproperty.construction_perc.alias("RealEstateConstructionStatus"),
            prproperty.land_ownership.cast("string").alias("RealEstateLandOwnership"),
            prproperty_building.name.alias("RealEstateBuildingName"),
            when(prproperty_building.active == 1, "Active")
            .when(prproperty_building.active == 0, "Inactive")
            .otherwise("NULL")
            .alias("RealEstateBuildingActive"),
            prproperty_building.cad_no.alias("RealEstateBuildingLandRegistryUnit"),
            prproperty_building.cf_no.alias("RealEstateBuildingLandBookNumber"),
            prproperty_building.old_cad_no.alias(
                "RealEstateBuildingOldLandRegistryUnit"
            ),
            lit(None).cast("String").alias("RealEstatePropertyBuildingNumber"),
            prproperty_building.old_cf_no.alias("RealEstateBuildingOldLandBookNumber"),
            prproperty_building.previous_cad.alias(
                "RealEstateBuildingPreviousRegistryUnit"
            ),
            prproperty_building.topographic_no.alias(
                "RealEstateBuildingTopographicNumber"
            ),
            prproperty_building.asset_type.alias("RealEstateBuildingBuildingTypeCode"),
            prproperty_building.asset_type.alias("RealEstateBuildingBuildingType"),
            prproperty_building.descr.alias("RealEstateBuildingDescription"),
            prproperty_building.floor_id.cast("double").alias("RealEstateBuildingFloorRawValue"),
            prproperty_building.contract_area.alias("RealEstateBuildingContractArea"),
            prproperty_building.net_area.alias("RealEstateBuildingNetArea"),
            prproperty_building.fixed_area_2.alias("RealEstateBuildingSurfaceArea2"),
            prproperty_building.gla_area.alias("RealEstateBuildingGrossRentableArea"),
            prproperty_building.occupancy_rate.alias("RealEstateBuildingOccupancyRate"),
            prproperty_building.renovation_year.alias(
                "RealEstateBuildingRenovationYear"
            ),
            prproperty_building.permit_no.alias("RealEstateBuildingPermitNumber"),
            prproperty_building.permit_date.alias("RealEstateBuildingPermitDate"),
            prproperty_building.tte_facade.alias("RealEstateBuildingFacadeArea"),
            prproperty_building.tte_freeheight.alias("RealEstateBuildingFreeHeight"),
            prproperty_building.tte_technicalnotes.alias(
                "RealEstateBuildingTechnicalNotes"
            ),
            prproperty_building.tte_otherauxiliaryarea.alias(
                "RealEstateBuildingOtherAuxiliaryArea"
            ),
            prproperty_building.tte_parkingspaces.alias(
                "RealEstateBuildingParkingSpace"
            ),
            prproperty_building.tte_storagearea.alias("RealEstateBuildingStorageArea"),
            prproperty_building.tte_loftarea.alias("RealEstateBuildingLoftArea"),
            prproperty_building.tte_PropertyDetachment.alias(
                "RealEstateBuildingDetachmentArea"
            ),
            prproperty_building.power_no.alias("RealEstateBuildingPowerNumber"),
            prproperty_building.date_entered.alias("RealEstateBuildingDateEntered"),
            prproperty_building.entered_by.alias("RealEstateBuildingEnteredBy"),
            prproperty_building.area_title.alias("RealEstateBuildingAreaTitle"),
            prproperty_building.area_license.alias("RealEstateBuildingAreaLicense"),
            prproperty_building.area_settled.alias("RealEstateBuildingAreaSettled"),
            prproperty_building.usage.alias("RealEstateBuildingUsageCode"),
            prproperty_building.parking_area.alias("RealEstateBuildingParkingArea"),
            prproperty_building.warehouse_no.alias("RealEstateBuildingWarehouseNumber"),
            prproperty.eligible_property.alias("RealEstateIsEligibleFlag"),
            prproperty.country_code.cast("string").alias("RealEstateCountry"),
            lit(None).cast("string").alias("CreatedDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("string").alias("LastUpdatedDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            prproperty.location_tracking.cast("string").alias("LocationTrackingCode"),
            target_table_thesis.LocationTracking.cast("string").alias("LocationTracking"),
            prproperty.businessdate.alias("BusinessDate"),
            lit(None).cast("string").alias("RealEstateBuildingFloorRawValueDescription"),

        )

        # ================== Joins for temp ==================
        
        prpropertytemp_prpropertybuildingtemp_join_condition = (
            thesis_prpropertytemp["ID"] == thesis_prpoperty_building_temp["property_temp_id"]
        )

        target_table_thesis_res = thesis_prpropertytemp.join(
            thesis_prpoperty_building_temp, 
            prpropertytemp_prpropertybuildingtemp_join_condition, 
            "left"
        )


        prproperty_building_primary_key_fields = [
            thesis_prpoperty_building_temp.ID,
            thesis_prpropertytemp.ID,
            lit("thesis_RES"),
        ]

        RealEstateBuildingKey_primary_key_fields = [
            thesis_prpoperty_building_temp.ID,
            thesis_prpropertytemp.ID,
        ]

        target_table_thesis_res = (
            target_table_thesis_res.withColumn(
                "RealEstateBuildingId",
                sha2(concat_ws(";", *prproperty_building_primary_key_fields), 256),
            )
            .withColumn(
                "RealEstateBuildingKey",
                concat_ws(";", *RealEstateBuildingKey_primary_key_fields),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_thesis_res = target_table_thesis_res.select(
            target_table_thesis_res.RealEstateBuildingId,
            lit("thesis_RES").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table_thesis_res.RealEstateBuildingKey,
            lit("Real Estate participate in the Real Estate Research").alias("RealEstateBuildingType"),
            thesis_prpropertytemp.ID.cast("string").alias("RealEstateKey"),
            lit(None).alias("RealEstateCepalKey"),
            thesis_prpropertytemp.status_code.alias("RealEstateStatusCode"),
            thesis_prpropertytemp.status_code.alias("RealEstateStatus"),
            thesis_prpropertytemp.table_name.alias("SourceTableName"),
            thesis_prpropertytemp.comments.alias("RealEstateComments"),
            lit(None).alias("RealEstateActive"),
            thesis_prpropertytemp.descr.alias("RealEstateDescription"),
            thesis_prpropertytemp.cat_code.alias("RealEstateSubCategoryCode"),
            thesis_prpropertytemp.cat_code.alias("RealEstateSubCategory"),
            lit(None).alias("RealEstateCategoryCode"),
            lit(None).alias("RealEstateCategory"),
            lit(None).cast("string").alias("RealEstateInsuranceCategory"),
            thesis_prpropertytemp.address.alias("RealEstateAddress1"),
            thesis_prpropertytemp.address_2.alias("RealEstateAddress2"),
            thesis_prpropertytemp.building_no.alias("RealEstateBuildingNumber"),
            lit(None).alias("RealEstateEntranceAddress"),
            thesis_prpropertytemp.zip.alias("RealEstatePostalCode"),
            lit(None).alias("RealEstateGeolocation"),
            thesis_prpropertytemp.location_lambda.alias("RealEstateLocationLambda"),
            thesis_prpropertytemp.location_phi.alias("RealEstateLocationPhi"),
            lit(None).alias("RealEstateEncumbrances"),
            lit(None).alias("RealEstateEncumbrancesStatus"),
            lit(None).alias("RealEstateEncumbrancesDetails"),
            thesis_prpropertytemp.land_area.alias("RealEstateLandArea"),
            lit(None).alias("RealEstateContractArea"),
            thesis_prpoperty_building_temp.net_area.alias("RealEstateNetArea"),
            lit(None).alias("RealEstateGlaArea"),
            lit(None).cast("string").alias("RealEstateOccupancyTypeCode"),
            lit(None).alias("RealEstateOccupancyType"),
            lit(None).alias("RealEstateOccupancyRate"),
            lit(None).alias("RealEstateSurfaceArea"),
            lit(None).alias("RealEstatePlottedArea"),
            lit(None).alias("RealEstatePlottedDate"),
            lit(None).alias("RealEstateAcquisitionValue"),
            lit(None).alias("RealEstateBookValue"),
            lit(None).alias("RealEstateApprovedIASValue"),
            lit(None).alias("RealEstateFiscalValue"),
            lit(None).alias("RealEstateMarketValue"),
            lit(None).alias("RealEstateEvaluationDate"),
            lit(None).alias("RealEstateEvaluationTypeCode"),
            lit(None).alias("RealEstateEvaluationType"),
            lit(None).alias("RealEstateForcedSaleValue"),
            lit(None).alias("RealEstateSaleAmount"),
            lit(None).alias("RealEstateSaleVATAmount"),
            lit(None).alias("RealEstateSaleTotalAmount"),
            lit(None).alias("RealEstateSaleDate"),
            thesis_prpropertytemp.date_entered.cast("date").alias("RealEstateDateEntered"),
            lit(None).alias("RealEstateFirstOfferPrice"),
            lit(None).alias("RealEstateMinimumPrice"),
            lit(None).alias("RealEstateMaximumPrice"),
            lit(None).alias("RealEstateAuctionAmount"),
            lit(None).alias("RealEstateConstructionPercentage"),
            lit(None).alias("RealEstateLegalIssuesFlag"),
            lit(None).alias("RealEstateValueDate"),
            lit(None).alias("RealEstateGroupMarketValueDate"),
            lit(None).alias("RealEstateValueTypeCode"),
            lit(None).alias("RealEstateValueType"),
            lit(None).alias("RealEstateGroupMarketValue"),
            lit(None).alias("RealEstateGroupMarketValueTypeCode"),
            lit(None).alias("RealEstateGroupMarketValueType"),
            lit(None).alias("RealEstateConstructionYear"),
            thesis_prpropertytemp.kaek.alias("RealEstateLandRegistryCode"),
            lit(None).alias("RealEstateLandRegistryOffice"),
            lit(None).alias("RealEstateLandRegistry"),
            thesis_prpropertytemp.usage_code.alias("RealEstateUsageCodeTypeCode"),
            lit(None).alias("RealEstateUsageCodeType"),
            lit(None).alias("RealEstateSiteCode"),
            lit(None).alias("RealEstatePrimaryLandLocation"),
            lit(None).alias("RealEstateTotalTax"),
            lit(None).alias("RealEstateOperatingCost"),
            thesis_prpropertytemp.area.alias("RealEstateArea"),
            lit(None).alias("RealEstateCity"),
            lit(True).alias("RealEstateIsEAPFlag"),
            lit(None).alias("RealEstateDistrict"),
            lit(None).alias("RealEstateDivision"),
            lit(None).alias("RealEstateMunicipality"),
            lit(None).alias("RealEstateCounty"),
            lit(None).alias("RealEstateMarketabilityCode"),
            lit(None).alias("RealEstateKaekCode"),
            lit(None).alias("RealEstateKaekOffice"),
            lit(None).alias("RealEstateHotelFlag"),
            lit(None).alias("RealEstateHotelStars"),
            lit(None).alias("RealEstateHotelRooms"),
            lit(None).alias("RealEstateSpecialConditions"),
            lit(None).alias("RealEstateAuctionDate"),
            lit(None).alias("RealEstateAuctionCode"),
            lit(None).alias("RealEstatePlaceCode"),
            lit(None).alias("RealEstateConstructionStatusCode"),
            lit(None).alias("RealEstateConstructionStatus"),
            lit(None).cast("string").alias("RealEstateLandOwnership"),
            lit(None).alias("RealEstateBuildingName"),
            lit(None).alias("RealEstateBuildingActive"),
            lit(None).alias("RealEstateBuildingLandRegistryUnit"),
            lit(None).alias("RealEstateBuildingLandBookNumber"),
            lit(None).alias(
                "RealEstateBuildingOldLandRegistryUnit"
            ),
            lit(None).cast("String").alias("RealEstatePropertyBuildingNumber"),
            lit(None).alias("RealEstateBuildingOldLandBookNumber"),
            lit(None).alias(
                "RealEstateBuildingPreviousRegistryUnit"
            ),
            lit(None).alias(
                "RealEstateBuildingTopographicNumber"
            ),
            lit(None).alias("RealEstateBuildingBuildingTypeCode"),
            lit(None).alias("RealEstateBuildingBuildingType"),
            lit(None).alias("RealEstateBuildingDescription"),
            thesis_prpoperty_building_temp.floor_id.cast("double").alias("RealEstateBuildingFloorRawValue"),
            lit(None).alias("RealEstateBuildingContractArea"),
            lit(None).alias("RealEstateBuildingNetArea"),
            lit(None).alias("RealEstateBuildingSurfaceArea2"),
            lit(None).alias("RealEstateBuildingGrossRentableArea"),
            lit(None).alias("RealEstateBuildingOccupancyRate"),
            lit(None).alias(
                "RealEstateBuildingRenovationYear"
            ),
            lit(None).alias("RealEstateBuildingPermitNumber"),
            lit(None).alias("RealEstateBuildingPermitDate"),
            lit(None).alias("RealEstateBuildingFacadeArea"),
            lit(None).alias("RealEstateBuildingFreeHeight"),
            lit(None).alias(
                "RealEstateBuildingTechnicalNotes"
            ),
            thesis_prpoperty_building_temp.tte_otherauxiliaryarea.alias(
                "RealEstateBuildingOtherAuxiliaryArea"
            ),
            lit(None).alias(
                "RealEstateBuildingParkingSpace"
            ),
            thesis_prpoperty_building_temp.tte_storagearea.alias("RealEstateBuildingStorageArea"),
            lit(None).alias("RealEstateBuildingLoftArea"),
            lit(None).alias(
                "RealEstateBuildingDetachmentArea"
            ),
            lit(None).alias("RealEstateBuildingPowerNumber"),
            thesis_prpoperty_building_temp.date_entered.alias("RealEstateBuildingDateEntered"),
            thesis_prpoperty_building_temp.entered_by.alias("RealEstateBuildingEnteredBy"),
            lit(None).alias("RealEstateBuildingAreaTitle"),
            lit(None).alias("RealEstateBuildingAreaLicense"),
            lit(None).alias("RealEstateBuildingAreaSettled"),
            lit(None).alias("RealEstateBuildingUsageCode"),
            thesis_prpoperty_building_temp.parking_area.alias("RealEstateBuildingParkingArea"),
            lit(None).alias("RealEstateBuildingWarehouseNumber"),
            lit(None).alias("RealEstateIsEligibleFlag"),
            lit(None).cast("string").alias("RealEstateCountry"),
            lit(None).cast("string").alias("CreatedDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("string").alias("LastUpdatedDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("string").alias("LocationTrackingCode"),
            lit(None).cast("string").alias("LocationTracking"),
            thesis_prpropertytemp.BusinessDate.alias("BusinessDate"),
            lit(None).cast("string").alias("RealEstateBuildingFloorRawValueDescription"),
        )

        # ================== Union ==================

        # Union the DataFrames
        final_real_estate_building_df = result_df_thesis.union(
            result_df_thesis_res
        )

        # ================== Append ==================

        final_real_estate_building_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_building_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)